In [3]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

In [4]:
load_dotenv()

True

In [5]:
# Global variable to store document content 
document_content = ""

In [6]:
class AgentState(TypedDict): 
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [7]:
@tool
def update(content: str) -> str:
    """Updates the document with the provided content."""
    global document_content
    document_content = content
    return f"Document has been updated successfully! The current content is:\n{document_content}"

@tool
def save(filename: str) -> str:
    """
    Save the current document to a text file and finish the process

    Args:
        filename: Name for the text file
    """

    global document_content

    if (not filename.endswith(".txt")):
        filename = f"{filename}.txt"

    try:
        with open(filename, "w") as file:
            file.write(document_content)
        print(f"Document has been saved successfully to {filename}")
        return f"Document has been saved successfully to '{filename}.'"

    except Exception as e: 
        # the return value passes into ToolMessage, which later, is used in the should_continue check - "document"
        return f"Error saving document: {e}"

In [8]:
tools = [update, save]

model = ChatOpenAI(model="gpt-5-mini").bind_tools(tools)

In [ ]:
def run_agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content=f"""
    You are Drafter, a helpful writing assistant. You are going to help the user update and modify documents.
    
    - If the user wants to update or modify content, use the 'update' tool with the complete updated content.
    - If the user wants to save and finish, you need to use the 'save' tool.
    - Make sure to always show the current document state after modifications.

    The current document content is: {document_content}
    """)

    if not state["message"]:
        user_input = input("\nWhat would you like to create in the document? ")
        user_message = HumanMessage(content=user_input)
    else: 
        user_input = input("\nWhat would you like to do with the document? ")
        print(f"\nUSER: {user_input}")
        user_message = HumanMessage(content=user_input)

    all_messages = [system_prompt] + list[state["messages"]] + [user_message]

    response = model.invoke(all_messages)

    print(f"\nAI: {response.content}")

    if hasattr(response, "tool_calls") and response.tool_calls: 
        print(f"USING TOOLS: {[tc["name"] for tc in response.tool_calls]}")

    return {"messages": list(state["messages"]) + [user_message, response]}